# 📊 Bitcoin Trader Behavior Insights
## Exploring the Relationship Between Market Sentiment & Trader Performance on Hyperliquid

> **Assignment:** Junior Data Scientist — Trader Behavior Insights | PrimeTrade.ai  
> **Dataset:** 211,224 trades from 32 traders (May 2023 – May 2025) merged with Bitcoin Fear & Greed Index

---

### 🔍 Objectives
1. Understand how Bitcoin market sentiment (Fear/Greed) influences trader performance
2. Identify hidden patterns in trade behavior across sentiment regimes
3. Cluster traders into behavioral archetypes
4. Build an ML model to predict profitable trades using sentiment as a feature
5. Derive actionable trading strategy insights


## 0. Environment Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
import warnings
warnings.filterwarnings('ignore')

# ── Dark theme palette ─────────────────────────────────────────────────────
PALETTE = {
    'Extreme Fear': '#d32f2f', 'Fear': '#ff7043',
    'Neutral': '#ffd600', 'Greed': '#66bb6a', 'Extreme Greed': '#1565c0',
}
SENTIMENT_ORDER = ['Extreme Fear', 'Fear', 'Neutral', 'Greed', 'Extreme Greed']
BG, CARD, TEXT = '#0d1117', '#161b22', '#e6edf3'
ACCENT, GREEN, RED, YELLOW = '#58a6ff', '#3fb950', '#f85149', '#d29922'

plt.rcParams.update({
    'figure.facecolor': BG, 'axes.facecolor': CARD,
    'axes.edgecolor': '#30363d', 'axes.labelcolor': TEXT,
    'xtick.color': TEXT, 'ytick.color': TEXT,
    'text.color': TEXT, 'grid.color': '#21262d',
    'grid.linestyle': '--', 'grid.alpha': 0.5,
})
print("✅ Setup complete")

## 1. Data Loading & Preprocessing

We load two datasets:
- **Historical Trader Data** from Hyperliquid (211K rows, 32 traders)
- **Bitcoin Fear & Greed Index** (daily sentiment, 2018–2025)

We merge them on date to create a unified dataset for analysis.


In [ ]:
# Load datasets
df = pd.read_csv('historical_data.csv')
fg = pd.read_csv('fear_greed_index.csv')

# Parse dates
df['datetime'] = pd.to_datetime(df['Timestamp IST'], dayfirst=True, errors='coerce')
df['date']     = df['datetime'].dt.date.astype(str)
fg['date']     = pd.to_datetime(fg['date']).dt.date.astype(str)

# Merge on date
df = df.merge(fg[['date','value','classification']], on='date', how='left')
df.rename(columns={'value':'fg_value', 'classification':'sentiment'}, inplace=True)

# Identify closed trades (trades with realized PnL)
df['is_closed'] = df['Direction'].isin(['Close Long', 'Close Short'])
closed = df[df['is_closed']].copy()
closed['profit'] = closed['Closed PnL'] > 0
closed['trade_type'] = np.where(closed['Direction']=='Close Long','Long','Short')

# Apply categorical ordering
df['sentiment']     = pd.Categorical(df['sentiment'],     categories=SENTIMENT_ORDER, ordered=True)
closed['sentiment'] = pd.Categorical(closed['sentiment'], categories=SENTIMENT_ORDER, ordered=True)

print(f"Total trades     : {len(df):,}")
print(f"Closed trades    : {len(closed):,}")
print(f"Unique traders   : {df['Account'].nunique()}")
print(f"Date range       : {df['date'].min()} → {df['date'].max()}")
print(f"Sentiment coverage: {df['sentiment'].notna().mean()*100:.2f}%")
print(f"\nTop coins:\n{df['Coin'].value_counts().head(6).to_string()}")

In [ ]:
# Overview of closed trades
print("=== CLOSED TRADES SUMMARY ===")
print(f"Overall win rate      : {closed['profit'].mean()*100:.1f}%")
print(f"Avg PnL per trade     : ${closed['Closed PnL'].mean():.2f}")
print(f"Median PnL per trade  : ${closed['Closed PnL'].median():.2f}")
print(f"Total realized PnL    : ${closed['Closed PnL'].sum():,.0f}")
print(f"\nLong trades  : {(closed['trade_type']=='Long').sum():,}")
print(f"Short trades : {(closed['trade_type']=='Short').sum():,}")

## 2. Exploratory Data Analysis

### 2.1 Fear & Greed Sentiment Distribution


In [ ]:
from IPython.display import Image
Image('fig1_sentiment_distribution.png')

**Key observations:**
- The market was in **Fear** for the most days (~30%), reflecting Bitcoin's historically volatile and uncertain nature
- **Extreme Greed** is the rarest state — bull runs are short-lived
- Combined Fear states (Fear + Extreme Fear) account for ~49% of all days


### 2.2 PnL Performance by Sentiment

In [ ]:
Image('fig2_pnl_by_sentiment.png')

**Insights:**
- Traders earn the **highest average PnL during Fear** periods (~$126/trade), likely because oversold conditions create high-quality entry opportunities
- **Extreme Greed** produces the lowest average PnL ($46/trade) — overheated markets lead to choppier, less directional trades
- The total cumulative PnL is dominated by **Fear and Neutral** regimes simply because more trading occurs in those periods


### 2.3 Win Rate by Sentiment

In [ ]:
Image('fig3_winrate_by_sentiment.png')

**Insights:**
- Highest win rate: **Fear (88.6%)** — traders are most precise when markets are fearful
- Lowest win rate: **Greed (76.1%)** — euphoria-driven markets produce more false breakouts
- Interestingly, **Extreme Greed** shows a recovery in win rate (87.4%), possibly due to experienced traders shorting tops decisively
- All sentiment categories maintain win rates well above 50%, indicating this trader cohort is highly skilled


### 2.4 Coin Performance Heatmap

In [ ]:
Image('fig4_heatmap_coin_sentiment.png')

**Insights:**
- **HYPE** consistently generates high PnL across all sentiment regimes — a high-alpha coin
- **BTC and ETH** perform better during Fear — classic "buy the dip" behavior
- Some coins (e.g., FARTCOIN, MELANIA) show sentiment-specific concentration, indicating speculative positioning


### 2.5 Long vs Short Performance by Sentiment

In [ ]:
Image('fig5_long_short_sentiment.png')

**Insights:**
- **Long trades outperform shorts during Fear** — confirming that buying fear is more profitable
- **Short trades are more effective during Greed** — sophisticated traders fade rallies
- This validates a classic counter-trend strategy: be long during fear, short during greed


### 2.6 Time Series: PnL vs Fear & Greed Index

In [ ]:
Image('fig6_timeseries.png')

**Insights:**
- Cumulative PnL shows a **strong upward trend** across the full 2-year period
- Notable PnL acceleration aligns with extreme fear zones — consistent with contrarian profitability
- The Fear/Greed index clearly cycles, and traders who adapt to these cycles sustain profits


### 2.7 PnL Distribution by Sentiment (Violin)

In [ ]:
Image('fig9_violin_pnl.png')

**Insights:**
- **Fear** sentiment shows the widest distribution — more variance, but more upside
- **Neutral** is tightest — predictable, moderate outcomes
- All regimes have positive median PnL (above the break-even line), confirming consistent profitability


## 3. Trader Segmentation (K-Means Clustering)

We cluster the 32 traders using behavioral features:
- Total & average PnL
- Win rate
- Number of trades (activity level)
- Average position size
- PnL volatility (std dev)


In [ ]:
Image('fig7_clustering.png')

**4 Trader Archetypes Identified:**

| Cluster | Label | Characteristics |
|---------|-------|-----------------|
| 🏆 | **Elite Performers** | High win rate (>50%), high total PnL, consistent returns |
| ⚡ | **High Frequency** | Many trades, moderate PnL per trade, volume-driven |
| 🔵 | **Moderate Traders** | Balanced activity, average performance |
| 📉 | **Struggling Traders** | Negative total PnL, lower win rates, possibly over-leveraged |

> Elite Performers tend to trade more actively during Fear, confirming the sentiment-performance link.


## 4. Machine Learning: Predicting Profitable Trades

**Goal:** Can we predict whether a trade will be profitable using sentiment + trade features?

**Features used:**
- `Size USD` — position size
- `fg_value` — Fear & Greed index value (0–100)
- `sentiment_enc` — encoded sentiment class
- `trade_type_enc` — Long (1) or Short (0)
- `hour` — hour of day (IST)
- `day_of_week` — day of week

**Target:** `profit` = 1 if Closed PnL > 0, else 0


In [ ]:
Image('fig8_ml_model.png')

**Results:**

| Model | Accuracy | ROC-AUC |
|-------|----------|---------|
| Random Forest | 86.5% | **89.2%** |
| Gradient Boosting | 88.3% | **89.2%** |
| Logistic Regression | 83.5% | 61.9% |

**Key Findings:**
- Tree-based models achieve **~89% AUC** — sentiment IS predictive of trade success
- `fg_value` and `sentiment_enc` are among the **top feature importances**, validating the hypothesis
- `Size USD` is the most important feature — larger positions tend to be more decisive/profitable
- `hour` of day also matters — certain market sessions are more favorable


## 5. Key Insights & Strategy Recommendations

### 📌 Insight 1: Fear is the Best Time to Trade
Traders achieve **88.6% win rate and $126 avg PnL** during Fear periods.  
→ **Strategy:** Increase position sizing and trade frequency during Fear sentiment.

### 📌 Insight 2: Greed is the Danger Zone
Win rate drops to **76.1%** and avg PnL halves during Greed.  
→ **Strategy:** Reduce long exposure, consider contrarian shorts during Greed periods.

### 📌 Insight 3: Long During Fear, Short During Greed
Long trades outperform during Fear; Shorts outperform during Greed.  
→ **Strategy:** Implement a sentiment-aware directional bias switch.

### 📌 Insight 4: Sentiment is a Predictive Signal
ML models using sentiment features achieve **89.2% AUC**.  
→ **Strategy:** Integrate Fear & Greed index as a real-time filter in trade execution.

### 📌 Insight 5: Coin Selection Matters by Regime
HYPE outperforms in all regimes; BTC/ETH are best fear plays.  
→ **Strategy:** Build a dynamic coin watchlist that updates with sentiment shifts.

---

## 6. Conclusion

This analysis demonstrates a **statistically meaningful relationship** between Bitcoin market sentiment and trader profitability on Hyperliquid. The Fear & Greed index is not just a lagging indicator — it is a **leading signal** for trade quality when used correctly.

The recommended framework is a **Sentiment-Adaptive Trading System**:
1. Monitor daily Fear & Greed classification
2. Switch directional bias (Long-heavy in Fear, Short-heavy in Greed)
3. Adjust position sizing based on sentiment confidence
4. Prioritize high-alpha coins (HYPE, BTC, ETH) aligned to the regime

> Built with Python · pandas · scikit-learn · matplotlib · seaborn
